# 01 数据管道 — 从手工档到自动档

**核心问题**：Notebook 00 里我们写了 fetch_baostock() 拉数据、手动清洗列名、转换类型。如果以后有 10 个策略、3 个市场、50 只股票，每次都要重复这些操作吗？

**本课答案**：建立**统一数据层**——写一次，到处用。

---

## 你将学到

| 章节 | 概念 | 技能 |
|------|------|------|
| 1. 问题回顾 | 重复代码的代价 | 从 Notebook 00 提炼痛点 |
| 2. 统一 Schema | 数据契约 | 所有市场输出同样的列 |
| 3. ABC 抽象基类 | 多态、接口分离 | 定义 DataSource 契约 |
| 4. AShareSource | baostock 工程化封装 | 批量拉取 + 错误隔离 |
| 5. CryptoSource | 跨市场统一接口 | 用同一套代码访问加密市场 |
| 6. SQLite 存储 | WAL + UPSERT | 数据落地与增量更新 |
| 7. 端到端验证 | 集成测试 | 拉取→存储→读取→画图 |

## 1. 从 Notebook 00 的痛点出发

回忆一下我们在 Notebook 00 里做了什么：

In [ ]:
# === Notebook 00 的做法（每次都要写这些） ===
import baostock as bs
import pandas as pd
import time

def fetch_baostock(symbol, start, end):
    bs.login()
    try:
        sy, ey = int(start[:4]), int(end[:4])
        all_rows = []
        for y in range(sy, ey + 1):
            seg_start = start if y == sy else f"{y}-01-01"
            seg_end   = end   if y == ey else f"{y}-12-31"
            rs = bs.query_history_k_data_plus(
                symbol, "date,open,high,low,close,volume,amount",
                start_date=seg_start, end_date=seg_end, frequency="d", adjustflag="2")
            rows = []
            while rs.next():
                rows.append(rs.get_row_data())
            if rows: all_rows.extend(rows)
            time.sleep(0.3)
        df = pd.DataFrame(all_rows, columns=["date","open","high","low","close","volume","amount"])
        for c in ["open","high","low","close","volume","amount"]:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        df["date"] = pd.to_datetime(df["date"])
        return df.drop_duplicates("date").sort_values("date").reset_index(drop=True)
    finally:
        bs.logout()

# 每拉一次数据就要调一次这个函数
df_index = fetch_baostock("sh.000300", "2024-01-01", "2024-06-30")
df_moutai = fetch_baostock("sh.600519", "2024-01-01", "2024-06-30")
# 如果要拉 50 只股票？要写 50 行，或者手动 for 循环
print(f"CSI 300: {len(df_index)} days, Moutai: {len(df_moutai)} days")

**痛点总结**：
- 每拉一次数据都要重复 login/logout/类型转换/去重逻辑
- 无法批量拉取（要自己写 for 循环）
- 拉回来的数据没有 `market` / `interval` 标记，混在一起分不清
- 如果想支持加密市场？得重新写一套完全不同的代码

**解决思路**：把所有"重复的"和"通用的"抽到一个基类，每个市场只写"差异的"部分。

## 2. 统一 Schema — 所有市场的共同语言

就像不同国家的人说不同语言，需要统一翻译成英语才能交流。不同数据源返回不同格式的数据，需要统一成**标准 OHLCV Schema**。

```
baostock 返回:  [date, open, high, low, close, volume, amount]
ccxt 返回:      [timestamp, open, high, low, close, volume]
yfinance 返回:  [Date, Open, High, Low, Close, Volume]
                         ↓
        统一 Schema（10 列标准）
     [symbol, date, open, high, low, close, volume, amount, market, interval]
```

这个 Schema 就是整个系统的**数据契约**——任何人提供数据都必须遵守这 10 列的格式。

In [ ]:
# 导入统一 Schema（已写在 data/schema.py）
import sys
sys.path.insert(0, "..")

from data.schema import OHLCV_COLUMNS, OHLCV_DTYPES

print("标准 OHLCV 列（10 列）：")
for i, col in enumerate(OHLCV_COLUMNS):
    print(f"  {i+1}. {col:12s} → {OHLCV_DTYPES[col]}")

print(f"\n每一行代表：某个品种在某个交易日的一根K线")
print(f"symbol + date = 唯一标识一行")
print(f"market + interval = 区分来自哪个市场、什么周期")

## 3. ABC 抽象基类 — 为什么这是量化系统的骨架

### 3.1 先看"没有抽象层"的世界
```python
# 不用抽象层：每个市场写完全独立的代码
if market == "ashare":
    df = baostock_fetch(...)   # 这个函数有自己的参数格式
elif market == "crypto":
    df = ccxt_fetch(...)       # 这个函数参数完全不同
elif market == "usstock":
    df = yfinance_fetch(...)   # 又是另一种
# 每个分支返回的列名不同、类型不同 → 后面的代码无法统一处理
```

### 3.2 有抽象层的世界
```python
# 不管什么市场，同一个接口：get_history()
source = AShareSource()       # 也可以是 CryptoSource()
df = source.get_history(symbols, start, end)  # ← 完全相同的调用方式
# 返回的 DataFrame 100% 是标准 Schema → 后续代码不用改
```

这就是**多态 (Polymorphism)**：不同子类有同一个接口，上层代码不关心底层是谁。

In [ ]:
# 从抽象基类的源码理解设计思路
import inspect
from data.sources.base import DataSource

print("=" * 60)
print("DataSource 抽象基类 — 接口契约")
print("=" * 60)

# 查看基类定义了哪些抽象方法
for name, method in inspect.getmembers(DataSource, predicate=inspect.isfunction):
    if hasattr(method, '__isabstractmethod__') and method.__isabstractmethod__:
        sig = inspect.signature(method)
        print(f"\n  @abstractmethod")
        print(f"  def {name}{sig}")
        if method.__doc__:
            print(f"      {method.__doc__.strip()[:100]}")

print(f"\n\n框架自动完成（子类不用写）：")
print(f"  get_history() — 批量拉取 + 标准化输出")
print(f"  _normalize()  — 列名统一 + 类型转换 + 去重排序")
print(f"\n子类只需实现：")
print(f"  market 属性  — 返回 'ashare'/'crypto'/'usstock'")
print(f"  _fetch()     — 拉一只股票的原始数据")

## 4. AShareSource — 把 baostock 装进框架

然后对比 "旧代码 vs 新代码"的工作量差异。

In [ ]:
from data.sources.ashare import AShareSource

ashare = AShareSource()

# 对比：旧方式 vs 新方式
# ┌─────────────────────────────────────────────────────┐
# │ 旧方式（Notebook 00）                                │
# │ df1 = fetch_baostock("sh.000300", "2024-01-01", ...) │
# │ df2 = fetch_baostock("sh.600519", "2024-01-01", ...) │
# │ # 然后手动合并、手加 market 列、手排序...               │
# └─────────────────────────────────────────────────────┘
#
# ┌─────────────────────────────────────────────────────┐
# │ 新方式（AShareSource）                                │
# │ 一行搞定批量拉取 + 自动标准化                           │
# └─────────────────────────────────────────────────────┘

symbols = ["sh.000300", "sh.600519", "sz.300750"]
df_ashare = ashare.get_history(symbols, "2024-06-01", "2024-07-31")

print(f"一次调用拉取 {len(symbols)} 个品种")
print(f"总行数: {len(df_ashare)}")
print(f"列名: {df_ashare.columns.tolist()}")
print(f"market: {df_ashare['market'].unique()}")
print(f"\n每个 symbol 的数据量:")
print(df_ashare.groupby("symbol").size())
df_ashare.head()

In [ ]:
# 新增功能：错误隔离
# 如果某个 symbol 拉不到数据，不影响其他的
symbols_with_bad = ["sh.000300", "NOT.EXIST", "sh.600519"]
df = ashare.get_history(symbols_with_bad, "2024-06-01", "2024-06-30")

print(f"请求: {len(symbols_with_bad)}只 → 成功: {df['symbol'].nunique()}只")
print(f"失败的 symbol 被跳过，不影响其他数据")
print(f"symbols in result: {df['symbol'].unique()}")
df.head()

## 5. CryptoSource — 抽象层的终极价值

现在我们用**完全相同的接口**访问一个**完全不同的市场**——不需要学新 API。

In [ ]:
from data.sources.crypto import CryptoSource

crypto = CryptoSource()  # 默认 Binance

# 和 AShareSource 完全一样的调用方式！
# 唯一的区别：symbol 格式从 "sh.600519" 变成了 "BTC/USDT"
df_crypto = crypto.get_history(
    symbols=["BTC/USDT", "ETH/USDT"],
    start="2024-01-01",
    end="2024-06-30"
)

print(f"数据量: {len(df_crypto)} 行")
print(f"market: {df_crypto['market'].unique()}")
print(f"列名: {df_crypto.columns.tolist()}")
print(f"\n确认：列名与 AShareSource 输出完全一致")
print(f"  AShare: {df_ashare.columns.tolist()}")
print(f"  Crypto: {df_crypto.columns.tolist()}")
print(f"  → 后续所有分析代码可以直接复用，不需要改一行")
df_crypto.head()

In [ ]:
# 跨市场对比：用同一套代码分析两个市场的差异
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (df, title) in zip(axes, [
    (df_ashare[df_ashare.symbol=="sh.000300"], "沪深300"),
    (df_crypto[df_crypto.symbol=="BTC/USDT"],   "BTC/USDT")
]):
    ret = np.log(df["close"] / df["close"].shift(1)).dropna()
    ax.hist(ret, bins=50, density=True, alpha=0.7, color=["steelblue", "orange"][list(axes).index(ax)])
    
    mu, sig = ret.mean(), ret.std()
    x = np.linspace(mu - 4*sig, mu + 4*sig, 100)
    ax.plot(x, 1/(sig*np.sqrt(2*np.pi))*np.exp(-(x-mu)**2/(2*sig**2)), 'r-', linewidth=1)
    
    ax.set_title(f"{title}\n波动率={sig*100:.2f}% | 偏度={ret.skew():.2f} | 峰度={ret.kurtosis():.2f}")
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)

plt.suptitle("同一个分析代码，两个市场 — 抽象层的价值", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("观察：")
print("- BTC 波动率远超沪深300 → 加密市场风险更高")
print("- BTC 峰度更肥 → 极端行情比A股还频繁")
print("- 同一个分析 Notebook 的代码，换个数据源就能跑 — 这就是抽象层的价值")

## 6. SQLite 存储 — 数据落地

每次调用 API 拉数据太慢、太依赖网络、也对数据源不友好。好的做法：**拉到本地数据库，以后直接读**。

In [ ]:
from data.store import DataStore

store = DataStore("../data/quant.db")

# 存 A股数据
store.save(df_ashare, market="ashare", interval="daily")
print(f"ashare_daily 表已创建并写入 {len(df_ashare)} 行")

# 存 加密数据
store.save(df_crypto, market="crypto", interval="daily")
print(f"crypto_daily 表已创建并写入 {len(df_crypto)} 行")

In [ ]:
# 查看数据库里的表
print("数据库表清单：")
display(store.list_tables())

In [ ]:
# 从 SQLite 读回数据 — 不再调用 API！
df_loaded = store.load("ashare", "daily", symbols=["sh.000300"])

print(f"从 SQLite 加载: {len(df_loaded)} 行")
print(f"列名: {df_loaded.columns.tolist()}")

# 验证：读出来的和存进去的一致
original = df_ashare[df_ashare.symbol == "sh.000300"]
close_diff = abs(df_loaded["close"].sum() - original["close"].sum())
print(f"close 总和误差: {close_diff:.10f}  (应为0)")
df_loaded.head()

In [ ]:
# 关键设计：WAL 模式 + UPSERT 语义
import sqlite3
conn = sqlite3.connect("../data/quant.db")

# 验证 1: WAL 模式
wal = conn.execute("PRAGMA journal_mode").fetchone()[0]
print(f"WAL 模式: {wal}")
print(f"  → WAL (Write-Ahead Logging) 允许一边读一边写")
print(f"  → 不会出现 'database is locked' 错误")
print(f"  → 未来多个策略同时读数据不会互相阻塞")

# 验证 2: UPSERT
count_before = conn.execute("SELECT COUNT(*) FROM ashare_daily").fetchone()[0]
store.save(df_ashare, "ashare", "daily")  # 再存一次同样的数据
count_after = conn.execute("SELECT COUNT(*) FROM ashare_daily").fetchone()[0]
print(f"\nUPSERT: 重复写入后 {count_before} = {count_after} (不应翻倍)")

conn.close()

## 7. 端到端验证

完整链路：拉取 → 存储 → 读出 → 画图 → 确认数据完整

In [ ]:
# 从数据库读取已存储的数据，直接画 K 线
import matplotlib.dates as mdates

df_plot = store.load("ashare", "daily", symbols=["sh.000300"], start="2024-01-01")
df_plot = df_plot.sort_values("date")

fig, ax = plt.subplots(figsize=(14, 5))

# 净值曲线
nav = df_plot.set_index("date")["close"] / df_plot.set_index("date")["close"].iloc[0]
ax.plot(nav.index, nav.values, color="steelblue", linewidth=1.5)
ax.fill_between(nav.index, 1, nav.values, where=(nav.values >= 1), color='red', alpha=0.1)
ax.fill_between(nav.index, nav.values, 1, where=(nav.values < 1), color='green', alpha=0.1)

ax.set_title("沪深300 2024上半年净值曲线 (数据来源: SQLite → 无需联网)", fontsize=12)
ax.set_ylabel("净值")
ax.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3)

total_return = (nav.iloc[-1] - 1) * 100
print(f"2024上半年沪深300收益: {total_return:.2f}%")
print(f"数据行数: {len(df_plot)}")
print(f"数据来源: SQLite 本地数据库 — 断网也能画")
plt.show()

## 8. 总结：你刚刚完成了什么

| 能力 | 实现方式 |
|------|----------|
| 统一数据接口 | DataSource ABC → 所有市场同一个 get_history() |
| 批量拉取 + 错误隔离 | get_history 循环调 _fetch，失败不阻塞 |
| 跨市场数据 | AShareSource + CryptoSource → 同一套分析代码 |
| 本地存储 | SQLite WAL + UPSERT → 断网也能分析 |

### 架构对比

```
以前（Notebook 00）:
  拉数据: 每次手写 baostock login/query/logout
  清洗:   手动 rename、astype、dropna
  存储:   没有存，每次重拉
  跨市场: 完全不同的代码

现在（Notebook 01）:
  拉数据: source.get_history(symbols, start, end)  ← 一行
  清洗:   自动标准化（框架完成）
  存储:   store.save(df) / store.load()            ← 断网可用
  跨市场: 改一行: AShareSource() → CryptoSource()   ← 接口不变
```

### 下一步

有了稳定的数据管道，下一课我们构建**回测引擎**——让双均线策略真正跑在干净的数据上，自动计算所有绩效指标。